In [ ]:
# 버전 전처리 후 csv
import pandas as pd
import os
import glob
import gc

def filter_game_versions(target_folder):
    """
    폴더 내의 '_flattened.csv' 파일들을 찾아,
    gameVersion이 15 또는 16 버전인 데이터만 남기고 새로 저장합니다.
    """
    if not os.path.exists(target_folder):
        print(f"오류: 지정한 폴더 경로가 존재하지 않습니다.\n{target_folder}")
        return
        
    # 1. 평탄화가 완료된 파일들만 검색
    search_pattern = os.path.join(target_folder, "*_flattened.csv")
    file_list = glob.glob(search_pattern)
    
    if len(file_list) == 0:
        print(f"알림: 해당 폴더에 '_flattened.csv'로 끝나는 파일이 없습니다.\n{target_folder}")
        return
        
    print(f"총 {len(file_list)}개의 평탄화된 파일을 찾았습니다. 버전 필터링을 시작합니다.\n")
    print("=" * 60)
    
    for input_path in file_list:
        file_name = os.path.basename(input_path)
        
        # 새로운 저장 파일명 생성 (예: details_br1_part1_flattened_v15_16.csv)
        output_path = input_path.replace(".csv", "_v15_16.csv")
        
        try:
            print(f"▶ 전처리 시작: {file_name}")
            
            # 2. 데이터 불러오기
            df = pd.read_csv(input_path, low_memory=False)
            total_rows = len(df)
            
            if 'gameVersion' not in df.columns:
                print(f"  - [경고] 'gameVersion' 컬럼이 없어 건너뜁니다.")
                continue
                
            # 3. 버전 필터링 (15. 또는 16. 으로 시작하는지 검사)
            # 결측치(NaN)는 문자로 변환 시 'nan'이 되므로 필터링 과정에서 자연스럽게 제외됩니다.
            version_str = df['gameVersion'].astype(str)
            condition = version_str.str.startswith('15.') | version_str.str.startswith('16.')
            
            filtered_df = df[condition]
            filtered_rows = len(filtered_df)
            
            # 4. 결과 확인 및 저장
            if filtered_rows == 0:
                print(f"  - [알림] 15, 16 버전에 해당하는 데이터가 한 건도 없습니다.")
            else:
                filtered_df.to_csv(output_path, index=False, encoding='utf-8-sig')
                deleted_rows = total_rows - filtered_rows
                print(f"  - 성공! (총 {total_rows}행 중 -> {filtered_rows}행 보존 / {deleted_rows}행 삭제)")
                
            # 메모리 비우기
            del df
            del filtered_df
            gc.collect()
            
        except Exception as e:
            print(f"  - 실패! (사유: {e})")
            
    print("=" * 60)
    print("모든 파일의 버전 전처리 작업이 완료되었습니다!")

# ==========================================
# 실행을 위한 설정 영역
# ==========================================

# 파일들이 있는 폴더 지정
folder_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details"

# 함수 실행!
filter_game_versions(folder_path)

총 15개의 평탄화된 파일을 찾았습니다. 버전 필터링을 시작합니다.

▶ 전처리 시작: details_br1_part1_flattened.csv
  - 성공! (총 83740행 중 -> 63930행 보존 / 19810행 삭제)
▶ 전처리 시작: details_eun1_part1_flattened.csv
  - 성공! (총 101110행 중 -> 76840행 보존 / 24270행 삭제)
▶ 전처리 시작: details_euw1_part1_flattened.csv
  - 성공! (총 119910행 중 -> 94700행 보존 / 25210행 삭제)
▶ 전처리 시작: details_euw1_part2_flattened.csv
  - 성공! (총 119880행 중 -> 94750행 보존 / 25130행 삭제)
▶ 전처리 시작: details_jp1_part1_flattened.csv
  - 성공! (총 15990행 중 -> 14250행 보존 / 1740행 삭제)
▶ 전처리 시작: details_kr_part1_flattened.csv
  - 성공! (총 235080행 중 -> 174060행 보존 / 61020행 삭제)
▶ 전처리 시작: details_la1_part1_flattened.csv
  - 성공! (총 58040행 중 -> 45030행 보존 / 13010행 삭제)
▶ 전처리 시작: details_la2_part1_flattened.csv
  - 성공! (총 55220행 중 -> 41310행 보존 / 13910행 삭제)
▶ 전처리 시작: details_me1_part1_flattened.csv
  - 성공! (총 2730행 중 -> 2330행 보존 / 400행 삭제)
▶ 전처리 시작: details_na1_part1_flattened.csv
  - 성공! (총 92000행 중 -> 72290행 보존 / 19710행 삭제)
▶ 전처리 시작: details_oc1_part1_flattened.csv
  - 성공! (총 10910행 중 -> 8400행 보존 / 251

In [4]:
import pandas as pd
import os

def preprocess_lol_data(input_path, output_path):
    """
    리그오브레전드 상세 데이터를 분석하기 좋게 정제(전처리)하는 함수입니다.
    """
    if not os.path.exists(input_path):
        print(f"오류: '{input_path}' 파일이 존재하지 않습니다.")
        return None
        
    try:
        print("1. 데이터 불러오는 중...")
        df = pd.read_csv(input_path, low_memory=False)
        initial_rows = len(df)
        print(f" -> 원본 데이터 크기: {initial_rows}행, {len(df.columns)}열")
        
        # ---------------------------------------------------------
        # 전처리 1: 다시하기(Remake) 및 비정상 조기 종료 게임 제외 (15분 = 900초 기준)
        # ---------------------------------------------------------
        print("\n2. 15분 미만 조기 종료 게임(다시하기 등) 제외 중...")
        if 'gameDuration' in df.columns:
            df = df[df['gameDuration'] >= 900]
            remake_rows = initial_rows - len(df)
            print(f" -> {remake_rows}행 삭제 완료. (남은 데이터: {len(df)}행)")
        else:
            print(" -> 'gameDuration' 컬럼이 없어 건너뜁니다.")
            
        # ---------------------------------------------------------
        # 전처리 2: 모든 필드(컬럼)의 Null 값을 0으로 채우기 (요청 반영)
        # ---------------------------------------------------------
        print("\n3. 모든 데이터의 결측치(Null)를 0으로 변환 중...")
        df = df.fillna(0)
        print(" -> 전체 컬럼 결측치 0으로 처리 완료.")
            
        # ---------------------------------------------------------
        # 전처리 3: KDA 파생 변수 생성
        # KDA = (Kills + Assists) / Deaths (단, Deaths가 0이면 Kills + Assists로 계산)
        # ---------------------------------------------------------
        print("\n4. KDA 파생 변수 생성 중...")
        if all(col in df.columns for col in ['kills', 'deaths', 'assists']):
            # 데스가 0인 경우 분모가 0이 되어 무한대(inf) 에러가 발생하므로, 0을 1로 바꿔서 계산
            safe_deaths = df['deaths'].replace(0, 1)
            df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths
            # 소수점 둘째 자리까지 반올림
            df['KDA_calculated'] = df['KDA_calculated'].round(2)
            print(" -> 'KDA_calculated' 컬럼 추가 완료.")
        else:
            print(" -> 킬/데스/어시스트 컬럼을 찾을 수 없어 건너뜁니다.")
            
        # ---------------------------------------------------------
        # 완료 및 저장
        # ---------------------------------------------------------
        print(f"\n5. 전처리 완료된 데이터를 저장하는 중... ({output_path})")
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f" -> 저장 완료! 최종 데이터 크기: {len(df)}행, {len(df.columns)}열")
        
        return df

    except Exception as e:
        print(f"전처리 중 오류가 발생했습니다: {e}")
        return None

# ==========================================
# 실행을 위한 설정 영역
# ==========================================

# 입력 파일명
file_path_in = r'D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\details_vn2_part1_flattened_v15_16.csv'

# 출력 파일명
file_path_out = r'D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\회의 컬럼\details_vn2_part1_preprocessed.csv'

# 함수 실행
processed_data = preprocess_lol_data(file_path_in, file_path_out)

# 데이터가 잘 변환되었는지 KDA 등 일부 정보 확인
if processed_data is not None:
    display_cols = ['matchId', 'championName', 'kills', 'deaths', 'assists', 'KDA_calculated', 'gameDuration']
    display_cols = [col for col in display_cols if col in processed_data.columns]
    
    print("\n[전처리 결과 미리보기 (상위 5개)]")
    display(processed_data[display_cols].head())

1. 데이터 불러오는 중...
 -> 원본 데이터 크기: 56950행, 154열

2. 15분 미만 조기 종료 게임(다시하기 등) 제외 중...
 -> 10행 삭제 완료. (남은 데이터: 56940행)

3. 모든 데이터의 결측치(Null)를 0으로 변환 중...
 -> 전체 컬럼 결측치 0으로 처리 완료.

4. KDA 파생 변수 생성 중...
 -> 'KDA_calculated' 컬럼 추가 완료.

5. 전처리 완료된 데이터를 저장하는 중... (D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\회의 컬럼\details_vn2_part1_preprocessed.csv)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15316\3410998978.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths


 -> 저장 완료! 최종 데이터 크기: 56940행, 155열

[전처리 결과 미리보기 (상위 5개)]


,matchId,championName,kills,deaths,assists,KDA_calculated,gameDuration
0,VN2_780837097,Garen,9,5,3,2.40,1723
1,VN2_780837097,Ekko,8,9,15,2.56,1723
2,VN2_780837097,Anivia,5,4,9,3.50,1723
3,VN2_780837097,Swain,18,5,9,5.40,1723
4,VN2_780837097,Karma,5,9,21,2.89,1723


In [5]:
import pandas as pd
import os
import glob
import gc

def batch_preprocess_exclude_vn(input_folder, output_folder):
    """
    지정된 폴더에서 'vn'이 포함되지 않은 15_16 버전 파일들을 찾아,
    모든 결측치 0 변환 및 KDA 생성 등 일괄 전처리를 수행합니다.
    """
    if not os.path.exists(input_folder):
        print(f"오류: 입력 폴더 경로가 존재하지 않습니다.\n{input_folder}")
        return
        
    # 출력 폴더가 없다면 자동으로 생성
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"알림: 출력 폴더를 새로 생성했습니다. ({output_folder})")

    # 1. 전처리 대상 파일 검색 (v15_16 버전으로 평탄화된 파일들)
    search_pattern = os.path.join(input_folder, "*_flattened_v15_16.csv")
    all_files = glob.glob(search_pattern)
    
    # 2. 'vn'이 포함된 파일 제외
    target_files = [f for f in all_files if 'vn' not in os.path.basename(f).lower()]
    
    print(f"총 {len(all_files)}개의 파일 중 'vn'을 제외한 {len(target_files)}개의 파일 전처리를 시작합니다.\n")
    print("=" * 60)
    
    if len(target_files) == 0:
        print("전처리할 대상 파일이 없습니다.")
        return

    # 3. 대상 파일들을 순회하며 전처리 시작
    for input_path in target_files:
        file_name = os.path.basename(input_path)
        # 새로 저장할 파일명 (예: details_br1_part1_preprocessed.csv)
        new_file_name = file_name.replace("_flattened_v15_16.csv", "_preprocessed.csv")
        output_path = os.path.join(output_folder, new_file_name)
        
        try:
            print(f"▶ 전처리 진행 중: {file_name}")
            
            # 데이터 로드
            df = pd.read_csv(input_path, low_memory=False)
            initial_rows = len(df)
            
            # (1) 15분(900초) 미만 게임 제외
            if 'gameDuration' in df.columns:
                df = df[df['gameDuration'] >= 900]
                
            # (2) 모든 결측치(Null)를 0으로 채우기
            df = df.fillna(0)
            
            # (3) KDA 파생 변수 생성
            if all(col in df.columns for col in ['kills', 'deaths', 'assists']):
                safe_deaths = df['deaths'].replace(0, 1)
                df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths
                df['KDA_calculated'] = df['KDA_calculated'].round(2)
                
            # 결과 저장
            if len(df) == 0:
                print(f"  - [경고] 남은 데이터가 0행입니다. 저장을 생략합니다.")
            else:
                df.to_csv(output_path, index=False, encoding='utf-8-sig')
                print(f"  - 성공! (총 {initial_rows}행 -> {len(df)}행 정제) -> {new_file_name} 저장 완료")
            
            # 메모리 정리
            del df
            gc.collect()
            
        except Exception as e:
            print(f"  - 실패! (사유: {e})")
            
    print("=" * 60)
    print("모든 파일의 전처리 작업이 완료되었습니다!")

# ==========================================
# 실행을 위한 폴더 경로 설정
# ==========================================

# 원본 파일(_v15_16.csv)들이 모여있는 폴더 (이전 대화 기준)
input_folder_path = r'D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후'

# 전처리가 끝난 결과물을 저장할 폴더
output_folder_path = r'D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\회의 컬럼'

# 함수 실행!
batch_preprocess_exclude_vn(input_folder_path, output_folder_path)

총 15개의 파일 중 'vn'을 제외한 14개의 파일 전처리를 시작합니다.

▶ 전처리 진행 중: details_br1_part1_flattened_v15_16.csv


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15316\1179697458.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths


  - 성공! (총 63930행 -> 63930행 정제) -> details_br1_part1_preprocessed.csv 저장 완료
▶ 전처리 진행 중: details_eun1_part1_flattened_v15_16.csv


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15316\1179697458.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths


  - 성공! (총 76840행 -> 76830행 정제) -> details_eun1_part1_preprocessed.csv 저장 완료
▶ 전처리 진행 중: details_euw1_part1_flattened_v15_16.csv


...

  - 성공! (총 7940행 -> 7940행 정제) -> details_ru_part1_preprocessed.csv 저장 완료
▶ 전처리 진행 중: details_tr1_part1_flattened_v15_16.csv


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15316\1179697458.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths


  - 성공! (총 40050행 -> 40050행 정제) -> details_tr1_part1_preprocessed.csv 저장 완료
▶ 전처리 진행 중: details_tw2_part1_flattened_v15_16.csv


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15316\1179697458.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['KDA_calculated'] = (df['kills'] + df['assists']) / safe_deaths


  - 성공! (총 10880행 -> 10880행 정제) -> details_tw2_part1_preprocessed.csv 저장 완료
모든 파일의 전처리 작업이 완료되었습니다!


In [2]:
import pandas as pd
import os

def extract_champion_and_position(input_path, output_path):
    """
    원본 데이터에서 챔피언 이름(championName)과 라인(teamPosition)만 추출하여 저장합니다.
    """
    if not os.path.exists(input_path):
        print(f"오류: 파일을 찾을 수 없습니다.\n경로: {input_path}")
        return
        
    try:
        print("데이터를 불러오는 중입니다...")
        # 데이터 불러오기
        df = pd.read_csv(input_path, low_memory=False)
        
        # 필요한 두 컬럼만 선택
        # (만약 individualPosition을 원하시면 teamPosition 대신 individualPosition을 적어주세요)
        target_columns = ['championName', 'teamPosition']
        
        # 컬럼이 존재하는지 확인
        missing_cols = [col for col in target_columns if col not in df.columns]
        if missing_cols:
            print(f"오류: 다음 컬럼이 데이터에 없습니다: {missing_cols}")
            return
            
        # 데이터 추출
        extracted_df = df[target_columns]
        
        # 새로운 CSV 파일로 저장
        print("추출된 데이터를 저장하는 중입니다...")
        extracted_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        print(f"완료! 총 {len(extracted_df)}행의 데이터가 성공적으로 저장되었습니다.")
        print(f"저장 위치: {output_path}")
        
        # 미리보기 제공
        print("\n[추출된 데이터 미리보기]")
        display(extracted_df.head())
        
    except Exception as e:
        print(f"작업 중 오류가 발생했습니다: {e}")

# ==========================================
# 실행 영역
# ==========================================

# 입력 파일명 (올려주신 북미 전처리 데이터 기준)
input_csv = r'D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\15서렌 제외 null값 0으로 채움\details_kr_part1_preprocessed.csv'

# 출력 파일명 (원하시는 이름으로 변경 가능)
output_csv = r'D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\15서렌 제외 null값 0으로 채움\kr_part1_champion_position_only.csv'

# 함수 실행
extract_champion_and_position(input_csv, output_csv)

데이터를 불러오는 중입니다...
추출된 데이터를 저장하는 중입니다...
완료! 총 168930행의 데이터가 성공적으로 저장되었습니다.
저장 위치: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\15서렌 제외 null값 0으로 채움\kr_part1_champion_position_only.csv

[추출된 데이터 미리보기]


,championName,teamPosition
0,Mordekaiser,TOP
1,Elise,JUNGLE
2,Hwei,MIDDLE
3,Ashe,BOTTOM
4,TwistedFate,UTILITY
